### Allscripts Sunrise (SCM) - Procedure Occurrence Hydration

**STATUS: PLACEHOLDER - Waiting for source tables to be exposed**

**Required Tables (not yet available):**
- dbo.SXARCMAbstractProcedureDetail (procedure records)
- dbo.SXARCMAbstractProcedureDetailMapping (mapping/linkage)
- dbo.SXARCMAbstractCPTHPCPSDetail (CPT/HCPCS codes)
- dbo.SXAESEventProcedureCode (event-based procedures)
- dbo.SXAESEventProcedureCodeException (exceptions/exclusions)

**Strategy (when tables are available):**
- Map ClientGUID to person_id via source_to_person
- Extract CPT/HCPCS/ICD-10-PCS procedure codes
- Map to OMOP procedure concepts
- Link to visit_occurrence where possible
- Capture procedure dates and modifiers
- Handle procedure quantities and units

**Note:**
This notebook will be implemented once procedure tables are exposed in Sunrise BigQuery views.

In [0]:
source = 'allscripts_scm'

# Exploration Queries (Run when procedure tables become available)

In [0]:
# STEP 1: Identify the procedure table structure
# DESCRIBE _exponent._bronze_allscripts_scm_prod01_vw.[PROCEDURE_TABLE];

# STEP 2: Sample procedure data
# SELECT * FROM _exponent._bronze_allscripts_scm_prod01_vw.[PROCEDURE_TABLE] LIMIT 10;

# STEP 3: Check CPT/HCPCS code table
# DESCRIBE _exponent._bronze_allscripts_scm_prod01_vw.dbo_sxarcmabstractcpthpcpsdetail;
# SELECT * FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_sxarcmabstractcpthpcpsdetail LIMIT 10;

# STEP 4: Identify key fields:
# - Patient identifier (ClientGUID or similar)
# - Procedure code (CPT, HCPCS, ICD-10-PCS)
# - Procedure description/name
# - Procedure date/datetime
# - Procedure end date (for surgeries with duration)
# - Modifiers (CPT modifiers like -LT, -RT, -59)
# - Quantity (number of procedures performed)
# - Provider who performed procedure
# - Related visit or encounter

print("Run exploration queries above when procedure tables are available")

# Transformation Template

In [0]:
# TEMPLATE - Uncomment and adjust field names when procedure table is available
#
# silver_procedure_occurrence_df = spark.sql(f'''
# SELECT 
#   -- Required fields
#   source_to_person.person_id,
#   COALESCE(procedure_concept.omop_concept_id, 0) AS procedure_concept_id,
#   CAST(proc.ProcedureDate AS DATE) AS procedure_date,  -- ADJUST: Use actual procedure date field
#   proc.ProcedureDate AS procedure_datetime,
#   
#   -- Optional end date/time (for procedures with duration like surgeries)
#   CAST(proc.ProcedureEndDate AS DATE) AS procedure_end_date,  -- ADJUST: May be NULL for most procedures
#   proc.ProcedureEndDate AS procedure_end_datetime,
#   
#   -- Procedure type (32817 = EHR, 38000275 = EHR billing record, customize as needed)
#   32817 AS procedure_type_concept_id,
#   
#   -- CPT/HCPCS modifier (e.g., -LT for left side, -RT for right side, -59 for distinct procedure)
#   COALESCE(modifier_concept.omop_concept_id, 0) AS modifier_concept_id,  -- ADJUST: Map modifiers to OMOP
#   
#   -- Quantity (number of times procedure performed, usually 1)
#   COALESCE(proc.Quantity, 1) AS quantity,  -- ADJUST: Default to 1 if NULL
#   
#   -- Provider who performed the procedure
#   NULL AS provider_id,  -- TODO: Map to provider via ProviderGUID
#   
#   -- Link to visit where procedure occurred
#   NULL AS visit_occurrence_id,  -- TODO: Map to visit via VisitGUID
#   NULL AS visit_detail_id,
#   
#   -- Source values (preserve original codes)
#   cpt.CPTCode AS procedure_source_value,  -- ADJUST: May be CPT, HCPCS, or ICD-10-PCS
#   0 AS procedure_source_concept_id,  -- Usually 0 unless code is non-standard OMOP concept
#   
#   -- Modifier source value (preserve original modifier)
#   proc.ModifierCode AS modifier_source_value,  -- ADJUST: CPT modifier field
#   
#   -- Unique source identifier
#   CONCAT('{source}', ' | ', proc.GUID) AS procedure_occurrence_source_value,  -- ADJUST: Use primary key
#   '{source}' AS source_system
#   
# FROM _exponent._bronze_allscripts_scm_prod01_vw.[PROCEDURE_TABLE] proc  -- ADJUST: Actual table name
# 
# -- Join to CPT/HCPCS code table (if separate)
# LEFT JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_sxarcmabstractcpthpcpsdetail cpt
#   ON proc.ProcedureID = cpt.ProcedureID  -- ADJUST: Join condition
# 
# -- Join to person
# INNER JOIN _exponent.omop_mapping.source_to_person
#   ON CONCAT('{source}', CHAR(31), 'cv3client', CHAR(31), 'GUID', CHAR(31), CAST(proc.ClientGUID AS BIGINT)) = source_to_person.person_source_value
#   AND source_to_person.active_flag = TRUE
# 
# -- Map procedure codes to OMOP concepts (CPT, HCPCS, ICD-10-PCS)
# LEFT JOIN _exponent.omop_mapping.domain_source_to_concept procedure_concept
#   ON procedure_concept.source_id = cpt.CPTCode  -- ADJUST: Procedure code field
#   AND procedure_concept.domain_id = 'Procedure'
#   AND procedure_concept.source_system = '{source}'
# 
# -- Map modifiers to OMOP concepts
# LEFT JOIN _exponent.omop_mapping.domain_source_to_concept modifier_concept
#   ON modifier_concept.source_id = proc.ModifierCode  -- ADJUST: Modifier field
#   AND modifier_concept.domain_id = 'Modifier'
#   AND modifier_concept.source_system = '{source}'
# 
# WHERE proc.ProcedureDate IS NOT NULL  -- ADJUST: Procedure date field
#   AND cpt.CPTCode IS NOT NULL  -- ADJUST: Must have a procedure code
#   AND proc.Active = TRUE  -- ADJUST: Filter logic
# 
# LIMIT 10000  -- Remove limit for production
# ''')
# 
# display(silver_procedure_occurrence_df)
# silver_procedure_occurrence_df.createOrReplaceTempView("silver_procedure_occurrence")

print("Procedure occurrence transformation template - uncomment when tables available")

# Merge to Silver Template

In [0]:
# %sql
# MERGE INTO _exponent.omop_silver.procedure_occurrence AS t
# USING (
#   SELECT * FROM silver_procedure_occurrence 
#   WHERE procedure_date IS NOT NULL
# ) AS s
# ON t.procedure_occurrence_source_value = s.procedure_occurrence_source_value
# 
# WHEN MATCHED THEN UPDATE SET
#   t.person_id = s.person_id,
#   t.procedure_concept_id = s.procedure_concept_id,
#   t.procedure_date = s.procedure_date,
#   t.procedure_datetime = s.procedure_datetime,
#   t.procedure_end_date = s.procedure_end_date,
#   t.procedure_end_datetime = s.procedure_end_datetime,
#   t.procedure_type_concept_id = s.procedure_type_concept_id,
#   t.modifier_concept_id = s.modifier_concept_id,
#   t.quantity = s.quantity,
#   t.provider_id = s.provider_id,
#   t.visit_occurrence_id = s.visit_occurrence_id,
#   t.visit_detail_id = s.visit_detail_id,
#   t.procedure_source_value = s.procedure_source_value,
#   t.procedure_source_concept_id = s.procedure_source_concept_id,
#   t.modifier_source_value = s.modifier_source_value,
#   t.last_mod_tsp = CURRENT_TIMESTAMP()
# 
# WHEN NOT MATCHED THEN INSERT (
#   person_id,
#   procedure_concept_id,
#   procedure_date,
#   procedure_datetime,
#   procedure_end_date,
#   procedure_end_datetime,
#   procedure_type_concept_id,
#   modifier_concept_id,
#   quantity,
#   provider_id,
#   visit_occurrence_id,
#   visit_detail_id,
#   procedure_source_value,
#   procedure_source_concept_id,
#   modifier_source_value,
#   procedure_occurrence_source_value,
#   source_system,
#   last_mod_tsp
# )
# VALUES (
#   s.person_id,
#   s.procedure_concept_id,
#   s.procedure_date,
#   s.procedure_datetime,
#   s.procedure_end_date,
#   s.procedure_end_datetime,
#   s.procedure_type_concept_id,
#   s.modifier_concept_id,
#   s.quantity,
#   s.provider_id,
#   s.visit_occurrence_id,
#   s.visit_detail_id,
#   s.procedure_source_value,
#   s.procedure_source_concept_id,
#   s.modifier_source_value,
#   s.procedure_occurrence_source_value,
#   s.source_system,
#   CURRENT_TIMESTAMP()
# );

print("Silver merge template - uncomment when implementing")

# Populate Mapping Table Template

In [0]:
# %sql
# INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
#   procedure_occurrence_source_value,
#   active_flag
# )
# SELECT 
#   procedure_occurrence_source_value,
#   TRUE
# FROM _exponent.omop_silver.procedure_occurrence
# WHERE procedure_occurrence_source_value NOT IN (
#   SELECT procedure_occurrence_source_value 
#   FROM _exponent.omop_mapping.source_to_procedure_occurrence
#   WHERE active_flag = TRUE
# );

print("Mapping population template - uncomment when implementing")

# Merge to Gold Template

In [0]:
# %sql
# MERGE INTO _exponent.omop.procedure_occurrence AS gold
# USING (
#   SELECT 
#     source_to_procedure_occurrence.procedure_occurrence_id,
#     s.person_id,
#     s.procedure_concept_id,
#     s.procedure_date,
#     s.procedure_datetime,
#     s.procedure_end_date,
#     s.procedure_end_datetime,
#     s.procedure_type_concept_id,
#     s.modifier_concept_id,
#     s.quantity,
#     s.provider_id,
#     s.visit_occurrence_id,
#     s.visit_detail_id,
#     s.procedure_source_value,
#     s.procedure_source_concept_id,
#     s.modifier_source_value
#   FROM _exponent.omop_silver.procedure_occurrence s
#   JOIN _exponent.omop_mapping.source_to_procedure_occurrence
#     ON source_to_procedure_occurrence.procedure_occurrence_source_value = s.procedure_occurrence_source_value
#     AND source_to_procedure_occurrence.active_flag = TRUE
# ) AS src
# ON gold.procedure_occurrence_id = src.procedure_occurrence_id
# 
# WHEN MATCHED THEN UPDATE SET
#   gold.person_id = src.person_id,
#   gold.procedure_concept_id = src.procedure_concept_id,
#   gold.procedure_date = src.procedure_date,
#   gold.procedure_datetime = src.procedure_datetime,
#   gold.procedure_end_date = src.procedure_end_date,
#   gold.procedure_end_datetime = src.procedure_end_datetime,
#   gold.procedure_type_concept_id = src.procedure_type_concept_id,
#   gold.modifier_concept_id = src.modifier_concept_id,
#   gold.quantity = src.quantity,
#   gold.provider_id = src.provider_id,
#   gold.visit_occurrence_id = src.visit_occurrence_id,
#   gold.visit_detail_id = src.visit_detail_id,
#   gold.procedure_source_value = src.procedure_source_value,
#   gold.procedure_source_concept_id = src.procedure_source_concept_id,
#   gold.modifier_source_value = src.modifier_source_value
# 
# WHEN NOT MATCHED THEN INSERT (
#   procedure_occurrence_id,
#   person_id,
#   procedure_concept_id,
#   procedure_date,
#   procedure_datetime,
#   procedure_end_date,
#   procedure_end_datetime,
#   procedure_type_concept_id,
#   modifier_concept_id,
#   quantity,
#   provider_id,
#   visit_occurrence_id,
#   visit_detail_id,
#   procedure_source_value,
#   procedure_source_concept_id,
#   modifier_source_value
# )
# VALUES (
#   src.procedure_occurrence_id,
#   src.person_id,
#   src.procedure_concept_id,
#   src.procedure_date,
#   src.procedure_datetime,
#   src.procedure_end_date,
#   src.procedure_end_datetime,
#   src.procedure_type_concept_id,
#   src.modifier_concept_id,
#   src.quantity,
#   src.provider_id,
#   src.visit_occurrence_id,
#   src.visit_detail_id,
#   src.procedure_source_value,
#   src.procedure_source_concept_id,
#   src.modifier_source_value
# );

print("Gold merge template - uncomment when implementing")

# Domain Mapping Template

**When procedure tables become available, populate domain_source_to_concept:**

In [0]:
# Example INSERT for procedure concept mappings:
# 
# INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
#   source_system,
#   source_table,
#   source_field,
#   domain_id,
#   source_id,
#   source_value,
#   omop_concept_id,
#   active_flag,
#   last_update_tsp
# )
# SELECT 
#   'allscripts_scm',
#   'dbo_sxarcmabstractcpthpcpsdetail',
#   'CPTCode',
#   'Procedure',
#   cpt.CPTCode,
#   cpt.CPTDescription,
#   COALESCE(c.concept_id, 0),  -- Map to CPT/HCPCS/ICD-10-PCS concepts
#   1,
#   CURRENT_TIMESTAMP()
# FROM (
#   SELECT DISTINCT CPTCode, CPTDescription
#   FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_sxarcmabstractcpthpcpsdetail
#   WHERE CPTCode IS NOT NULL
# ) cpt
# LEFT JOIN _exponent.omop.concept c
#   ON c.concept_code = cpt.CPTCode
#   AND c.vocabulary_id IN ('CPT4', 'HCPCS', 'ICD10PCS')  -- Procedure vocabularies
#   AND c.domain_id = 'Procedure'
#   AND c.standard_concept = 'S';

print("Domain mapping template - adjust when procedure codes are known")

# Modifier Mapping Template (Optional)

**CPT modifiers (e.g., -LT, -RT, -59) may need separate mapping:**

In [0]:
# Example INSERT for modifier concept mappings:
# 
# INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
#   source_system,
#   source_table,
#   source_field,
#   domain_id,
#   source_id,
#   source_value,
#   omop_concept_id,
#   active_flag,
#   last_update_tsp
# )
# VALUES
#   ('allscripts_scm', '[PROCEDURE_TABLE]', 'ModifierCode', 'Modifier', 'LT', 'Left', 44785783, 1, CURRENT_TIMESTAMP()),
#   ('allscripts_scm', '[PROCEDURE_TABLE]', 'ModifierCode', 'Modifier', 'RT', 'Right', 44785784, 1, CURRENT_TIMESTAMP()),
#   ('allscripts_scm', '[PROCEDURE_TABLE]', 'ModifierCode', 'Modifier', '59', 'Distinct Procedural Service', 44785610, 1, CURRENT_TIMESTAMP());

print("Modifier mapping template - common CPT modifiers")